## Assignment 1

*100 points (7% of course grade)*</br>
*Assigned: Mon, Sep 15nd*</br>
**Due: Sun, Sep 28th, 23:59**

This homework should be done in parts as soon as (<= 1 week) relevant topics are covered in lectures. If you wait until the last minute, you might be overwhelmed.

You must turn in the required files electronically, including this Notebook (A1.ipynb) and a few additional files. Please follow the submission instructions for each problem carefully.

In this assignment, you need to solve two problems. In Problem 1, you will write relational algebra queries. In Problem 2, you will draw an E/R diagram.

Consider a database with following schema. Underlined columns are the keys of the table. 

- drinker (<u>name</u>, address)
- bar (<u>name</u>, address)
- beer (<u>name</u>, brewer)
- frequents (<u>drinker</u>, <u>bar</u>, times_a_week)
- likes (<u>drinker</u>, <u>beer</u>)
- serves (<u>bar</u>, <u>beer</u>, price)


## Setup environment and test database

You will need this setup if you want to create a database to test whether your answer is correct. Please follow our [setup instructions](https://canvas.sfu.ca/courses/91482/pages/0-main-entrance-general-guidance-on-cmpt-354-environment-setup) on Canvas. We recommend you finish the setup, as you can run your queries and debug on your machine, and you will need Postgres in future assignments, **but you are still able to finish Assignment 1 without the setup**.

### Problem 1: Query with Relational Algebra (63%)

#### **Preliminary**


To write a relational algebra (RA) query in a cell, we have already converted the cells under each question into the [Markdown cell](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html). Then, you will need to type RA queries in the cell using the syntax in [radb](https://users.cs.duke.edu/~junyang/radb/index.html). Below is an example of an RA query in the radb format. Please refer to [the radb cheatsheet](https://users.cs.duke.edu/~junyang/radb/cheat.html) for the full list of syntax of RA queries. Please use the radb's syntax to ensure we can run your RA queries. You may draw the query tree on your scratch paper and then replace the Greek letter operators with the radb-format operators.

Example: find the name beers liked by drinkers who frequent the James Joyce Pub bar.
```
\project_{beer} ( 
        (\select_{bar = 'James Joyce Pub'} Frequents)         
         \join_{Frequents.drinker = Likes.drinker}   /* join with Likes to find beers */
         Likes
);
```


You can run RA queries on your local machine after installing radb following the instructions on Canvas. We will also provide an online tool for you to debug your RA queries (stay tuned for the instructions). It will be much easier to grade if we can run your query and also easier for you to debug with query results. *The use of the tool is optional, and submissions to the online tool will not be graded (**we only grade the Canvas submission**).*



Now your homework question is to write Relational Algebra queries to answer following questions. 

Please fill your answer in each cell (and **ONLY the query**) and **DO NOT add or remove** any cells to make the TAs' life easier in evaluating your queries. Questions (1)-(3) are worth 6 points each; (4)-(6) are worth 7 points each; (7)-(9) are worth 8 points each.


#### 0. (example) Find names of all bars that Eve frequents.

/* input your answer in this cell: */

\project_{bar} (\select_{drinker = 'Eve'} frequents);

#### 1. Find names of beers that  Satisfaction serves

/* input your answer in this cell: */

\project_{beer} (\select_{bar ='Satisfaction'} serves);



#### 2. Find names of bars that Amy frequents more than once a week

/* input your answer in this cell: */

\project_{bar} (\select_{drinker = 'Amy' and times_a_week > 1} frequents);

#### 3. Find names of all drinkers who frequent at least two bars

/* input your answer in this cell: */

v1 :- \rename_{drinker1, bar1} (\project_{drinker, bar}(frequents));

v2 :- \rename_{drinker2, bar2} (\project_{drinker, bar}(frequents));

\project_{drinker1} (v1 \join_{drinker1 = drinker2 and bar1 <> bar2} v2);

#### 4. Find bars frequented by either Ben or Dan, but not both

/* input your answer in this cell: */

v1 :- \project_{bar} (\select_{drinker = 'Ben'} frequents);

v2 :- \project_{bar} (\select_{drinker = 'Dan'} frequents);

\project_{bar} (v1 \union v2) \diff (v1 \intersect v2);



#### 5. Find the names of all drinkers who frequent *every* bar (hint: you may need to use renaming and store your intermediate results using [views](https://users.cs.duke.edu/~junyang/radb/advance.html?highlight=view) to make your querying process more clear.)

e.g. with view, the first example query in preliminary can be written as

```
v1 :- \select_{bar = 'James Joyce Pub'} Frequents;
\project_{beer} ( 
        v1 
         \join_{v1.drinker = Likes.drinker}   /* join with Likes to find beers */
         Likes
);
```
For view names, please only use **lowercase** letters, because ratest may automatically convert letters to lowercase.

/* input your answer in this cell: */

v1 :- (\project_{name}(drinker)) \cross (\project_{name}(bar));

v2 :- \rename_{drinker, bar}(v1);

v3 :- v2 \diff \project_{drinker, bar}(frequents);

\project_{name}(drinker) \diff \project_{drinker}(v3);



#### 6. Find names and addresses of drinkers who like Corona but do not frequent Satisfaction

/* input your answer in this cell: */

v1 :- \rename_{name}(\project_{drinker}(\select_{beer = 'Corona'} (likes)));

v2 :- \rename_{name}(\project_{drinker}(\select_{bar = 'Satisfaction'} (frequents)));

v3 :- \rename_{name}((\project_{drinker}(frequents)) \diff (v2));

\project_{name, address}(
  (v1 \intersect v3) \join (drinker));


#### 7. For each beer that Eve likes, find the names of bars that serve it at the lowest price (when a bar serves multiple beers at the same lowest price, they should all be included in the output) (hint: recall the "trickier exercise" in the slides for how to present "the lowest")

/* input your answer in this cell: */

v1 :- \project_{beer}(\select_{drinker = 'Eve'}(likes));

v2 :- v1 \join serves;

v3 :- \rename_{beer1, bar1, price1}(v2);

v4 :- \rename_{beer2, bar2, price2}(v2);

v5 :- v3 \join_{beer1 = beer2 and price1 > price2} v4;

v6 :- \project_{beer1, bar1, price1}(v5);

v7 :- v2 \diff v6;

\project_{beer, bar}(v7);

#### 8. Find names of all drinkers who frequent *only* those bars that serve *some* beers they like (drinkers who frequent no bars are included)

/* input your answer in this cell: */

v1 :- \project_{name}(drinker);

v2 :- \project_{drinker, bar}(frequents);

v3 :- \project_{drinker, bar}((likes \join serves));

v4 :- \project_{drinker}(v2 \diff v3);

v1 \diff v4;

#### 9. For each beer, find the drinkers who like this beer but frequent *none* of the bars serving this beer. Format your output as a list of (beer, drinker) pairs.

/* input your answer in this cell: */

v1 :- \project_{beer, drinker}(likes);

v2 :- \project_{beer, drinker}(likes \join serves \join frequents);

v1 \diff v2;


## Problem 2: ER design (37%)

Design a database that captures the following information:
    
    
- Each person is either a student or teacher, but not both.


- Each person has a unique ID, a name, and phone (denoted by the model). 


- The university offers different courses of study. Each course has a unique name and belongs to a department. In any given school year, a given subject can be taught by only one teacher. A course can be taught over multiple years and a student may study the same course multiple times.


- For each student, you need to additionally record the year when he or she entered the university (the class year), as well as his or her favorite subjects.


- A student or a teacher can belong to one or multiple departments. You should be able to track each department's head and its current students.


- Each department has multiple clubs. You should be able to track each club's current students.


Design an E/R diagram for this database. Replace the current `ER-diagram.png` with your figure (you may also use other picture formats like .jpg, but remember to change the filename in the cell below). Very briefly explain the intuitive meaning of any entity and relationship sets as needed. Do not forget to indicate keys and multiplicity of relationships, as well as ISA relationships and weak entity sets (if any), using appropriate notation.


If you think some aspects of the above are unclear, feel free to make additional, reasonable assumptions, but state them clearly in your answer. Also, keep in mind that there is no single “correct” design.

<!-- <img src="ER-diagram.png" alt="Drawing" style="width: 800px;"/> -->
<img src="ER-diagram.svg" alt="Drawing" style="width: 800px;"/>

`Write a brief explanation for your design in this cell`
1. Entity sets:
- Person
- Students
- Teachers
- Departments
- Courses
- CourseOfferings (weak entity)
- Clubs
- All entity set have a key, denote by underlining

2. Notes/Assumptions about choice of relationships between entity sets:
- "Course has unique name"
 -> I assumed course name is globally unique (for example: CMPT354, CMPT225) -> let Course be a regular entity and belongs to Department. I assumed that a course can not be offered by multiple departments at the same time, so the relationship is many-to-one.

- For "In any given school year, a given subject can be taught by only one teacher (many-to-one). A course can be taught over multiple years and a student may study the same course multiple times (many-to-many).". 
-> I added a weak entity CourseOfferings with the 'year' attribute to indicate a specific course is offered in a special year (for example: CMPT354 Fall 2025). Also we can add other attributes like 'location', 'number of students enrolled' to CourseOfferings entity if we need to, so I think it's better choice compared to having a 'year' attribute to the relationship between Students/Teachers and Courses.

- I simply just added 'class year' as an attribute to Students entity instead of making it a relationship, because I don't see the necessity of creating another University entity set (I assumed we only have ONE university here so it wouldn't make sense if create an entity set). Meanwhile, I make 'favorite' a relationship because I already have Course entity sets.

- "You should be able to track each department's head"
-> I assumed that teacher can only be one head of a department, and department can only have one head at a time (one to one).
-> I made Person belongs to Departments -> Department can track current students. Or can track by: Students study the CourseOfferings (if current students mean active students).

- "Each department has multiple clubs. You should be able to track each club's current students."
-> I assumed that Club is unique within the university (not within the Departments). I also assumed that a club can only belong to one or zero department (many-to-one).

 

### Submission instruction

1. For problem 1, answer the questions (1)-(9) in the Markdown cells.

2. For problem 2, replace `ER-diagram.png` with your ER diagram in a png/jpg file; write some explanation in the Markdown cell.

3. Compress your A1.ipynb (this file) and your ER diagram into A1.zip and submit on Canvas.